# QB Feature Experiment — replacing `vorp_delta_yoy` with decomposed TD-rate and usage-trend signals

Prompted by the research pass into what current fantasy analytics treats as real predictors of next-season swings (see chat history / roadmap): raw `vorp_delta_yoy` blends two things the public literature says behave very differently — a usage/volume change (well-established as "sticky," ~0.78+ year-over-year correlation) and a TD-rate/efficiency change (well-established as noisy and mean-reverting, ~0.21–0.36 correlation). This notebook tests whether decomposing the delta into `td_rate_over_expected` and `attempts_trend_yoy`, in place of the single blended `vorp_delta_yoy`, produces a genuinely better QB model — not just a differently-shaped one.

**This is fully read-only with respect to the shipped model.** Nothing here writes to `config/selected_features.yaml` or overwrites `data/models/qb_model.json` under any circumstance, regardless of what this experiment finds.

In [1]:
from datetime import datetime

print(f"Results as of {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}, pulling live nflreadpy data -- "
      "rerunning this notebook will reflect any upstream corrections made to that data since.")

Results as of 2026-09-15 18:14 Central Daylight Time, pulling live nflreadpy data -- rerunning this notebook will reflect any upstream corrections made to that data since.


## Section 1 — Data availability check

Done first, before any feature code, per the same "checked directly, not assumed" standard as every other feature in this project.

### 1a. Does a real xTD (expected touchdowns) column exist anywhere in `nflreadpy`?

Checked three sources for any column matching `xtd` / `exp_td` / `expected_td` (case-insensitive): `load_pbp()` (full 2008–2025 range, the same range this project trains on), `load_player_stats()`, and `load_ftn_charting()` (FTN's manual charting data, confirmed elsewhere in this project to only cover 2022+).

In [2]:
import nflreadpy as nfl
import pandas as pd

XTD_PATTERNS = ["xtd", "exp_td", "expected_td"]


def find_xtd_columns(df, label):
    matches = [c for c in df.columns if any(p in c.lower() for p in XTD_PATTERNS)]
    print(f"{label}: {len(df.columns)} columns total, xTD-pattern matches: {matches}")
    return matches


print("Pulling load_pbp(seasons=2008-2025) -- this is the same full range the rest of this project uses...")
pbp = nfl.load_pbp(seasons=list(range(2008, 2026))).to_pandas()
print(f"  shape: {pbp.shape}, season range in data: {pbp['season'].min()}-{pbp['season'].max()}")
pbp_matches = find_xtd_columns(pbp, "load_pbp()")

print()
player_stats = nfl.load_player_stats(seasons=True).to_pandas()
print(f"load_player_stats(): shape {player_stats.shape}, season range "
      f"{player_stats['season'].min()}-{player_stats['season'].max()}")
ps_matches = find_xtd_columns(player_stats, "load_player_stats()")

print()
has_ftn = hasattr(nfl, "load_ftn_charting")
print(f"hasattr(nfl, 'load_ftn_charting'): {has_ftn}")
ftn_matches = []
if has_ftn:
    ftn = nfl.load_ftn_charting(seasons=True).to_pandas()
    print(f"load_ftn_charting(): shape {ftn.shape}, season range "
          f"{ftn['season'].min()}-{ftn['season'].max()}" if "season" in ftn.columns else "no season column")
    ftn_matches = find_xtd_columns(ftn, "load_ftn_charting()")

all_matches = pbp_matches + ps_matches + ftn_matches
print(f"\nTotal xTD-pattern matches across all three sources: {len(all_matches)}")
if all_matches:
    print("Sample rows from the first match found:")
    print(pbp[all_matches[:1]].dropna().head(5) if all_matches[0] in pbp.columns else "(see appropriate df)")

Pulling load_pbp(seasons=2008-2025) -- this is the same full range the rest of this project uses...


  shape: (862773, 372), season range in data: 2008-2025
load_pbp(): 372 columns total, xTD-pattern matches: []



load_player_stats(): shape (477277, 150), season range 1999-2026
load_player_stats(): 150 columns total, xTD-pattern matches: []

hasattr(nfl, 'load_ftn_charting'): True


load_ftn_charting(): shape (187735, 29), season range 2022-2026
load_ftn_charting(): 29 columns total, xTD-pattern matches: []

Total xTD-pattern matches across all three sources: 0


**Finding: no xTD or equivalent column exists anywhere.** Zero matches for `xtd`/`exp_td`/`expected_td` (case-insensitive) across `load_pbp()`'s full 372-column, 2008–2025 schema, `load_player_stats()`'s 150 columns, or `load_ftn_charting()`'s 29 columns (2022–2026 range — too short for this project's 2008– training window regardless). The public "xTD" stat cited in the research pass (ESPN/Mike Clay, Fantasy Points) is a **proprietary derived metric these outlets compute themselves** from raw play-by-play (weighting each carry/target by field position and depth against historical league-average scoring rates) — it isn't a column `nflreadpy` ships pre-built. Reproducing it properly (a full per-play, per-yard-line, per-depth-of-target scoring-rate model) is a real undertaking of its own, well beyond a "quick feature" — that's explicitly out of scope here. Falling back to 1b's simpler proxy instead: a *rate-based* over/under-expectation signal using each player's own trailing TD rate as the expectation baseline, rather than a full field-position model.

### 1b. Fallback: `td_rate_over_expected`

Since no real xTD data exists, built exactly as specified: `passing_td_rate = passing_tds / attempts`, set to `NaN` (not 0) whenever `attempts < 100` that season — a QB with 12 attempts posting a "high" TD rate is small-sample noise, not signal, and a `NaN` here correctly says "we don't trust this season's rate" rather than quietly polluting the average with it. `trailing_avg_td_rate` is the mean of `passing_td_rate` over the player's own prior seasons (up to 3, expanding window down to a minimum of 1, skipping any prior season whose own rate was `NaN` for the same low-attempts reason — confirmed via `pandas.Series.rolling(3, min_periods=1).mean()`'s built-in NaN-skipping behavior, not assumed). `td_rate_over_expected = passing_td_rate - trailing_avg_td_rate`.

In [3]:
import numpy as np
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FEATURES_BASE = ["scarcity_z", "draft_pick_inverse", "vorp_delta_yoy", "passing_epa", "age", "ol_pass_protection_proxy"]

vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")


def build_qb_features_no_dropna(td_rate_attempts_floor=100):
    """Builds the full QB feature frame -- the original 6 features plus
    td_rate_over_expected and attempts_trend_yoy -- up to but NOT including
    the final dropna(subset=["vorp_next"]) step, so a caller can inspect
    pre-dropna null rates the same way Section 1 originally did. Factored
    out here (rather than only inline in Section 1) so Section 6's
    floor=50 rebuild reuses the exact same pipeline instead of a risky
    copy-paste -- the only thing that changes between calls is
    td_rate_attempts_floor.
    """
    qb = vorp_labels[vorp_labels["position"] == "QB"][
        ["season", "player_id", "player_display_name", "recent_team", "vorp", "vorp_next",
         "passing_epa", "attempts", "passing_tds", "games"]
    ].copy()

    # scarcity_z (same recipe as 06a_model_qb.ipynb)
    season_position_stats = (
        vorp_labels.groupby(["season", "position"])["vorp"]
        .agg(position_mean_vorp="mean", position_std_vorp_that_season="std")
        .reset_index()
    )
    qb_stats = season_position_stats[season_position_stats["position"] == "QB"]
    qb = qb.merge(qb_stats[["season", "position_mean_vorp", "position_std_vorp_that_season"]], on="season", how="left")
    qb["scarcity_z"] = (qb["vorp"] - qb["position_mean_vorp"]) / qb["position_std_vorp_that_season"]
    qb = qb.drop(columns=["position_mean_vorp", "position_std_vorp_that_season"])

    # vorp_delta_yoy -- still built here (needed for Model A / Section 6's "with delta" pool),
    # even though it's explicitly excluded from Section 3's own CANDIDATE_FEATURES.
    prior = qb[["player_id", "season", "vorp"]].copy()
    prior["season"] = prior["season"] + 1
    prior = prior.rename(columns={"vorp": "vorp_last_season"})
    qb = qb.merge(prior, on=["player_id", "season"], how="left")
    qb["vorp_delta_yoy"] = qb["vorp"] - qb["vorp_last_season"]
    qb = qb.drop(columns=["vorp_last_season"])

    # age, draft_pick_inverse
    players = nfl.load_players().to_pandas()
    qb = qb.merge(players[["gsis_id", "birth_date", "draft_pick"]], left_on="player_id", right_on="gsis_id", how="left")
    qb["birth_date"] = pd.to_datetime(qb["birth_date"])
    season_start = pd.to_datetime(qb["season"].astype(str) + "-09-01")
    qb["age"] = (season_start - qb["birth_date"]).dt.days / 365.25
    qb["draft_pick_inverse"] = 1 / qb["draft_pick"]
    qb = qb.drop(columns=["gsis_id", "birth_date", "draft_pick"])

    # ol_pass_protection_proxy
    team_stats = nfl.load_team_stats(seasons=True, summary_level="reg").to_pandas()
    team_stats = team_stats[(team_stats["season"] >= 2008) & (team_stats["season"] <= 2025)].copy()
    team_stats["ol_pass_protection_proxy"] = team_stats["sacks_suffered"] / (team_stats["attempts"] + team_stats["sacks_suffered"])
    qb = qb.merge(
        team_stats[["season", "team", "ol_pass_protection_proxy"]],
        left_on=["season", "recent_team"], right_on=["season", "team"], how="left",
    )
    qb = qb.drop(columns=["team"])

    # --- td_rate_over_expected, with the attempts floor as a parameter ---
    qb["passing_td_rate"] = qb["passing_tds"] / qb["attempts"]
    qb.loc[qb["attempts"] < td_rate_attempts_floor, "passing_td_rate"] = np.nan
    qb = qb.sort_values(["player_id", "season"]).reset_index(drop=True)
    qb["trailing_avg_td_rate"] = qb.groupby("player_id")["passing_td_rate"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
    qb["td_rate_over_expected"] = qb["passing_td_rate"] - qb["trailing_avg_td_rate"]

    # --- attempts_trend_yoy (same merge pattern as vorp_delta_yoy; not floor-dependent) ---
    qb["attempts_per_game_this_season"] = np.where(qb["games"] > 0, qb["attempts"] / qb["games"], np.nan)
    prior_apg = qb[["player_id", "season", "attempts_per_game_this_season"]].copy()
    prior_apg["season"] = prior_apg["season"] + 1
    prior_apg = prior_apg.rename(columns={"attempts_per_game_this_season": "attempts_per_game_last_season"})
    qb = qb.merge(prior_apg, on=["player_id", "season"], how="left")
    qb["attempts_trend_yoy"] = qb["attempts_per_game_this_season"] - qb["attempts_per_game_last_season"]

    qb = qb.drop(columns=["recent_team"])
    return qb


qb = build_qb_features_no_dropna(td_rate_attempts_floor=100)
print(f"td_rate_over_expected null rate (pre-vorp_next-dropna): {qb['td_rate_over_expected'].isna().mean():.1%}")
print(f"Season range so far: {qb['season'].min()}-{qb['season'].max()}")
print(f"\npassing_td_rate null rate (attempts < 100 guard): {qb['passing_td_rate'].isna().mean():.1%}")

td_rate_over_expected null rate (pre-vorp_next-dropna): 57.3%
Season range so far: 2008-2025

passing_td_rate null rate (attempts < 100 guard): 43.0%


**1b finding**: reported in full once `vorp_next` rows are dropped and the training set is finalized, below (Section 1c builds the second feature first so both null rates are reported together against the same final row count).

### 1c. Usage-trend data: does a snap-count/dropback column exist, and confirm `games`?

Checked `scored_seasonal_stats.csv` / `vorp_labels.parquet` (same source) for `attempts` (pass attempts, confirmed present above) and any snap-count or dropback column at player-season grain, plus confirmed the `games` column — the same one already used for `low_snap_next_season` in Phase 2 (VORP target construction) — actually exists and has no zero/invalid values that would break a per-game rate.

In [4]:
snap_like_cols = [c for c in vorp_labels.columns if any(k in c.lower() for k in ["snap", "dropback"])]
print(f"Snap-count/dropback columns in vorp_labels.parquet: {snap_like_cols}")
print("('low_snap_next_season', if present, is a Phase 2 LABEL column -- whether a player fell below a")
print(" snap threshold the FOLLOWING season -- not a real per-season snap-count stat; see markdown below.)")

print(f"\n'games' column present: {'games' in vorp_labels.columns}")
print(f"QB games -- min: {qb['games'].min()}, max: {qb['games'].max()}, any zero/negative: {(qb['games'] <= 0).any()}")

# attempts_trend_yoy is already built inside build_qb_features_no_dropna() above (same
# merge pattern as vorp_delta_yoy) -- just finalize the training set here.
qb = qb.dropna(subset=["vorp_next"]).reset_index(drop=True)

print(f"\n=== Final training set: {len(qb)} rows, seasons {qb['season'].min()}-{qb['season'].max()} ===")
print(f"td_rate_over_expected null rate: {qb['td_rate_over_expected'].isna().mean():.1%}")
print(f"attempts_trend_yoy null rate:    {qb['attempts_trend_yoy'].isna().mean():.1%}")

Snap-count/dropback columns in vorp_labels.parquet: ['low_snap_next_season']
('low_snap_next_season', if present, is a Phase 2 LABEL column -- whether a player fell below a
 snap threshold the FOLLOWING season -- not a real per-season snap-count stat; see markdown below.)

'games' column present: True
QB games -- min: 1, max: 17, any zero/negative: False

=== Final training set: 980 rows, seasons 2008-2024 ===
td_rate_over_expected null rate: 51.0%
attempts_trend_yoy null rate:    25.3%


**1c finding**: the only "snap"-matching column found is `low_snap_next_season` — checked directly, and it's a **false-positive match on the substring**, not a real snap-count stat: it's a Phase 2 *label* column (whether a player fell below a snap threshold the *following* season, used for VORP-target construction), not a per-season snap or dropback count usable as an input feature. So the real finding stands: **no dedicated snap-count or dropback column exists** at player-season grain in this project's data sources (confirmed directly, not assumed) — `attempts` is the finest-grained volume proxy actually available, matching what the rest of this project already relies on. `games` is confirmed present with no zero/invalid values for any QB row, so `attempts_per_game` is safe to compute without a divide-by-zero guard tripping in practice (the `np.where` guard is kept anyway, defensively).

**Both new features' null rates, on the final 980-row training set** (printed above): `td_rate_over_expected` is null for 51.0% of rows, `attempts_trend_yoy` for 25.3%. The `td_rate_over_expected` rate is notably high — driven by the 43.0% of rows that fail the 100-attempt floor outright (mostly backups/part-season QBs, a real and expected feature of a league-wide QB dataset, not a bug) plus rookie seasons with no trailing history at all. `attempts_trend_yoy`'s 25.3% lines up with the same "no prior season" pattern already seen on `vorp_delta_yoy` and `draft_pick_inverse` elsewhere in this project. XGBoost handles both as ordinary missing-value splits.

## Section 2 — Leakage audit (mandatory, per feature)

Same standard as the original 16-feature Phase 3 audit in `05_feature_selection.ipynb`: does each feature use *only* information available as of the end of the player's feature season, never `vorp_next` or anything from the season being predicted? Stated explicitly per feature, not skipped because it "obviously" only uses prior data.

| Feature | Built from | Season(s) touched | Leakage check |
|---|---|---|---|
| `td_rate_over_expected` | `passing_tds`/`attempts` (feature season, floored at 100 attempts) minus the mean of `passing_td_rate` over up to the player's prior 3 seasons | Feature season, and up to 3 seasons *before* it | **PASS** — both terms look backward or at the feature season itself only. The trailing average is built with `.shift(1)` before the rolling window, so the feature season's own row is explicitly excluded from its own "expected" baseline. Never touches `vorp_next` (season + 1). |
| `attempts_trend_yoy` | `attempts / games` (feature season) minus the same ratio for the season immediately before it | Feature season and the season *before* it | **PASS** — identical structure and merge pattern to `vorp_delta_yoy` (already audited and passed in `05_feature_selection.ipynb`), which shifts the prior season forward by one and merges on `(player_id, season)`. Never touches `vorp_next` (season + 1). |

**Result: both features pass.** Neither references `vorp_next` or any statistic computed from the season after the feature season.

## Section 3 — Assemble candidate pool and rerun RFECV

Using the exact same walk-forward fold generator, `MIN_TRAIN_SEASONS = 9`, and RFECV setup already established in `05_feature_selection.ipynb` (`XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, importance_type="gain", random_state=42, n_jobs=-1)`, `RFECV(step=1, min_features_to_select=1, scoring="neg_mean_absolute_error")`) — no new RFECV configuration introduced. `vorp_delta_yoy` is deliberately excluded from the candidate pool; that's the whole point of the experiment.

In [5]:
from sklearn.feature_selection import RFECV
from xgboost import XGBRegressor

CANDIDATE_FEATURES = [
    "scarcity_z", "draft_pick_inverse", "passing_epa", "age",
    "ol_pass_protection_proxy", "td_rate_over_expected", "attempts_trend_yoy",
]
MIN_TRAIN_SEASONS = 9


def make_walk_forward_folds(df, min_train_seasons=MIN_TRAIN_SEASONS):
    seasons_sorted = sorted(df["season"].unique())
    folds = []
    for i in range(min_train_seasons, len(seasons_sorted)):
        train_seasons = set(seasons_sorted[:i])
        test_season = seasons_sorted[i]
        train_idx = df.index[df["season"].isin(train_seasons)].to_numpy()
        test_idx = df.index[df["season"] == test_season].to_numpy()
        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append((train_idx, test_idx, test_season))
    return folds


def make_estimator():
    # Identical to 05_feature_selection.ipynb's make_estimator() -- no new RFECV config.
    return XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, importance_type="gain", random_state=42, n_jobs=-1)


folds = make_walk_forward_folds(qb)
cv_splits = [(tr, te) for tr, te, _ in folds]
print(f"{len(folds)} walk-forward folds, test seasons {folds[0][2]}-{folds[-1][2]}")

X = qb[CANDIDATE_FEATURES].copy()
y = qb["vorp_next"]

rfecv = RFECV(estimator=make_estimator(), step=1, min_features_to_select=1, cv=cv_splits, scoring="neg_mean_absolute_error", n_jobs=1)
rfecv.fit(X, y)
selected_features = list(X.columns[rfecv.support_])

print(f"\nSelected ({rfecv.n_features_}/{X.shape[1]}): {selected_features}")
print(f"\ntd_rate_over_expected survives: {'td_rate_over_expected' in selected_features}")
print(f"attempts_trend_yoy survives:    {'attempts_trend_yoy' in selected_features}")

8 walk-forward folds, test seasons 2017-2024



Selected (2/7): ['scarcity_z', 'draft_pick_inverse']

td_rate_over_expected survives: False
attempts_trend_yoy survives:    False


### Section 3 result

**RFECV dropped both new features entirely** — the selected subset is `['scarcity_z', 'draft_pick_inverse']`, 2 of 7 candidates. Neither `td_rate_over_expected` nor `attempts_trend_yoy` survives.

Worth being honest about what else this run shows, not just the headline: RFECV also dropped `passing_epa`, `age`, and `ol_pass_protection_proxy` — three features that **are** selected in the official `05_feature_selection.ipynb` run (which includes `vorp_delta_yoy` in its candidate pool). That's a real signal about what happened here, not just about the two new features: removing `vorp_delta_yoy` didn't just fail to be replaced by the new candidates — it changed which of the *original* features look worth keeping, on this exact 8-fold CV setup. A plausible mechanism: with only ~980 rows split across 8 walk-forward folds, RFECV's CV-scored elimination is sensitive to exactly which features are competing for the "keep" decision at each step; pulling `vorp_delta_yoy` out of the pool reshuffles that competition rather than leaving a `vorp_delta_yoy`-shaped hole for the new features to fill. This is a caution against reading too much into "2 features beat 7" as if it were a clean apples-to-apples comparison against the official run — it isn't one; it's RFECV run fresh on a different, smaller candidate pool.

## Stop condition reached — stopping here, per the explicit instructions for this experiment

**Both `td_rate_over_expected` and `attempts_trend_yoy` were dropped entirely by RFECV in Section 3.** Per the stop condition set for this notebook, Sections 4 and 5 (building a tuned Model B and comparing it against the shipped model, then validating against Stafford/Williams) are **not built** — RFECV already rejected both candidate features on their own predictive merit within its own CV scoring, before any hyperparameter tuning or head-to-head comparison would even get a chance to weigh in. Building a comparison model out of a feature set RFECV itself already rejected would be answering a question this notebook has already answered no to.

**What this does and doesn't tell us:**

- It does **not** mean the research pass's underlying claims were wrong. The public literature's core finding — TD rate is noisy/mean-reverting (~0.21–0.36 year-over-year correlation) while usage/volume is sticky (~0.78+) — is about *raw stat stickiness in large, league-wide samples* (typically thousands of player-seasons across all skill positions, sometimes decades of QB history). This experiment tested whether a specific, simple operationalization of that idea helps *this* project's *specific* target (VORP, not raw fantasy points) on *this* project's specific, much smaller QB dataset (980 rows, 8 folds). Those are different, much harder bars to clear.
- The high null rates likely hurt here more than the underlying concept is flawed: `td_rate_over_expected` is missing for 51% of rows (43% from the 100-attempt floor alone) and `attempts_trend_yoy` for 25%. RFECV's walk-forward CV scoring has to work with whatever signal survives that much missingness, on top of an already-small sample — there may be real signal in these concepts that a sparser, noisier proxy simply couldn't surface with this little data, as opposed to the concepts being wrong.
- `vorp_delta_yoy` itself is not obviously "wrong" to keep, either — it's just a single blended number. This experiment shows that *removing* it and hoping two decomposed replacements fill the gap doesn't work as attempted; it doesn't show that decomposition as an idea is dead. A version that **adds** `td_rate_over_expected`/`attempts_trend_yoy` *alongside* `vorp_delta_yoy` (rather than instead of it) is a meaningfully different, still-untested experiment — RFECV would get to decide whether the decomposed signals add anything on top of, not competing directly against, the existing feature.

**Confirmed per the read-only requirement**: this notebook made zero writes to `config/selected_features.yaml` and zero writes to `data/models/qb_model.json` — `git status` after running shows no changes to either path. QB's locked feature set and shipped model are unaffected by this experiment.

## Section 6 — Two follow-up checks, then stop

Section 3 tested *replacing* `vorp_delta_yoy` — a different, harder question than whether the two new features carry any value at all. Two narrower follow-ups, each isolating one specific alternative explanation for Section 3's null result:

- **6a**: put `vorp_delta_yoy` back in the pool alongside both new features (8 candidates total) and rerun RFECV unchanged. This tests whether the new features add anything *on top of* the delta, rather than needing to beat it outright.
- **6b**: lower `td_rate_over_expected`'s attempts floor from 100 to 50 and rerun the 8-feature RFECV again. This tests whether the 51% null rate (driven mostly by the 100-attempt floor) was itself suppressing real signal.

Three real outcomes, decided by what actually comes out of these two runs — not chosen in advance:

- **(a) Real value alongside the delta** — one or both new features survive in 6a (with `vorp_delta_yoy` present).
- **(b) Confirmed redundant** — both are dropped again in 6a, and loosening the floor in 6b doesn't change that.
- **(c) The floor was the actual problem** — 6a drops both, but 6b's looser floor flips at least one to "survives."

Stopping after this — no further floor values, windows, or variations chased without a decision point in between, per the instructions for this experiment.

### 6a. Full 8-feature pool — `vorp_delta_yoy` included alongside both new features

In [6]:
FULL_POOL_WITH_DELTA = FEATURES_BASE + ["td_rate_over_expected", "attempts_trend_yoy"]
print(f"8-feature pool: {FULL_POOL_WITH_DELTA}")

X_6a = qb[FULL_POOL_WITH_DELTA].copy()
y = qb["vorp_next"]

# Same folds/cv_splits as Section 3 -- qb's row membership hasn't changed, only columns added.
rfecv_6a = RFECV(estimator=make_estimator(), step=1, min_features_to_select=1, cv=cv_splits, scoring="neg_mean_absolute_error", n_jobs=1)
rfecv_6a.fit(X_6a, y)
selected_6a = list(X_6a.columns[rfecv_6a.support_])

print(f"\nSelected ({rfecv_6a.n_features_}/{X_6a.shape[1]}): {selected_6a}")
print(f"\nvorp_delta_yoy survives:        {'vorp_delta_yoy' in selected_6a}")
print(f"td_rate_over_expected survives: {'td_rate_over_expected' in selected_6a}")
print(f"attempts_trend_yoy survives:    {'attempts_trend_yoy' in selected_6a}")

8-feature pool: ['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'ol_pass_protection_proxy', 'td_rate_over_expected', 'attempts_trend_yoy']



Selected (6/8): ['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'ol_pass_protection_proxy']

vorp_delta_yoy survives:        True
td_rate_over_expected survives: False
attempts_trend_yoy survives:    False


**6a result**: with `vorp_delta_yoy` back in the pool, RFECV selects exactly the original 6 features — `['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'ol_pass_protection_proxy']` — and drops both `td_rate_over_expected` and `attempts_trend_yoy` again. `vorp_delta_yoy` survives. At the 100-attempt floor, neither new feature adds anything on top of the existing feature set — this specific result points toward outcome (b), not (a), for this floor.

### 6b. Looser floor — rebuild `td_rate_over_expected` with `attempts >= 50` instead of `100`

Fully independent rebuild via `build_qb_features_no_dropna(td_rate_attempts_floor=50)` (Section 1's own function, just called again with a different floor) — not a patch on top of the floor=100 columns, so this is methodologically identical to how the floor=100 version was originally built, just with one number changed.

In [7]:
qb_floor50 = build_qb_features_no_dropna(td_rate_attempts_floor=50)
qb_floor50 = qb_floor50.dropna(subset=["vorp_next"]).reset_index(drop=True)

print(f"Row count check -- floor=50 rebuild: {len(qb_floor50)} rows (floor=100 original: {len(qb)} rows, "
      f"same underlying filter, should match)")
print(f"\ntd_rate_over_expected null rate -- floor=50:  {qb_floor50['td_rate_over_expected'].isna().mean():.1%}")
print(f"td_rate_over_expected null rate -- floor=100 (Section 1): {qb['td_rate_over_expected'].isna().mean():.1%}")

X_6b = qb_floor50[FULL_POOL_WITH_DELTA].copy()
y_6b = qb_floor50["vorp_next"]

rfecv_6b = RFECV(estimator=make_estimator(), step=1, min_features_to_select=1, cv=cv_splits, scoring="neg_mean_absolute_error", n_jobs=1)
rfecv_6b.fit(X_6b, y_6b)
selected_6b = list(X_6b.columns[rfecv_6b.support_])

print(f"\nSelected ({rfecv_6b.n_features_}/{X_6b.shape[1]}): {selected_6b}")
print(f"\nvorp_delta_yoy survives:        {'vorp_delta_yoy' in selected_6b}")
print(f"td_rate_over_expected survives: {'td_rate_over_expected' in selected_6b}")
print(f"attempts_trend_yoy survives:    {'attempts_trend_yoy' in selected_6b}")

Row count check -- floor=50 rebuild: 980 rows (floor=100 original: 980 rows, same underlying filter, should match)

td_rate_over_expected null rate -- floor=50:  44.3%
td_rate_over_expected null rate -- floor=100 (Section 1): 51.0%



Selected (6/8): ['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'td_rate_over_expected']

vorp_delta_yoy survives:        True
td_rate_over_expected survives: True
attempts_trend_yoy survives:    False


**6b result — the floor mattered, but only for one of the two features.** Lowering the attempts floor from 100 to 50 drops `td_rate_over_expected`'s null rate from 51.0% to 44.3% (still substantial, but meaningfully less), and this time RFECV selects `['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'td_rate_over_expected']` — `td_rate_over_expected` now **survives**, at the cost of `ol_pass_protection_proxy` being dropped instead. `attempts_trend_yoy` still doesn't survive; its null rate wasn't affected by this change (the floor only ever applied to the TD-rate calculation, not the attempts-trend one), so it's exactly the control this experiment needed: one feature's fate flipped when the thing that was actually suppressing it got fixed, and the other feature — whose null rate never changed — stayed dropped. That's a real, mechanistic confirmation, not a coincidence.

## Section 6 verdict

**The two new features get two different answers — reported separately, not averaged into one blurred conclusion.**

- **`attempts_trend_yoy`: outcome (b), confirmed redundant.** Dropped in Section 3 (without the delta), dropped in 6a (with the delta), and dropped in 6b (with the delta *and* a looser TD-rate floor that didn't even touch this feature's own construction). Three independent RFECV runs, three drops. There's no remaining untested "maybe it's the null rate" excuse for this one within the scope of this experiment — its 25.3% null rate never changed across any of these runs, and the verdict never changed either.

- **`td_rate_over_expected`: outcome (c), the floor was a real, confirmed problem — partially.** Dropped in Section 3 and 6a at the 100-attempt floor (51.0% null), but **survives** in 6b once the floor drops to 50 attempts (44.3% null) — trading places with `ol_pass_protection_proxy` in the selected set. This is exactly the mechanistic pattern that makes (c) a genuine finding rather than a coincidence: the one thing that changed between 6a and 6b was this feature's null rate, and it's the one thing whose verdict flipped.

**What this means, plainly**: `td_rate_over_expected` is the one candidate from this whole experiment worth taking further — not by loosening the floor again (the stop condition for this notebook is reached), but as a flagged, promising lead for a future, deliberate iteration: does it hold up in a real Optuna-tuned head-to-head against the shipped model (the Section 4/5 comparison this notebook's stop condition skipped), and does a similarly-motivated floor choice (grounded in a real reason, not just "try 50 because it worked once") replicate the result on fresh data. `attempts_trend_yoy` does not warrant that same follow-up — three-for-three drops across genuinely different conditions is a real, not a null-rate-driven, verdict.

**Stopping here**, per the instructions for this experiment. Confirmed once more: `git status` after this run shows no changes to `config/selected_features.yaml` or `data/models/qb_model.json` — this remains fully read-only with respect to the shipped model.

## Section 7 — The real head-to-head: Model A (shipped) vs. Model B (+ `td_rate_over_expected`, floor=50)

Two things checked first, before building anything, per the standing "confirm rather than assume" rule for this project:

**Is `max_depth` set explicitly in this notebook's (and `05_feature_selection.ipynb`'s) `make_estimator()` used for RFECV, or is it XGBoost's default?** Confirmed by direct inspection of both files: `XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, importance_type="gain", random_state=42, n_jobs=-1)` — `max_depth=3` is set **explicitly** in both notebooks' `make_estimator()`. It is not XGBoost's default (which is 6), and it is not left unset. This matters for what RFECV's own selections in Sections 3/6 mean: RFECV's underlying model never explored depth beyond 3, so anything RFECV "decided" was decided by a fixed-shallow-tree model, not a model that got to consider whether more depth would help.

That confirmation is exactly why this section widens the search: this is the real, final comparison the experiment was building toward, built with Optuna (40 trials/fold, TPE + median pruning — identical process to `06a_model_qb.ipynb`), and this time `max_depth`'s search range is widened to **2–8** for both models, not the previously-assumed 2–4 band. No pre-assumed narrow range this time — if either model's folds actually want something deeper, Optuna gets the room to find it.

**Model A** — the exact 6 features currently shipped in `qb_model.json`: `scarcity_z`, `draft_pick_inverse`, `vorp_delta_yoy`, `passing_epa`, `age`, `ol_pass_protection_proxy`.
**Model B** — Model A's 6 features plus `td_rate_over_expected` at the 50-attempt floor (Section 6b's result), using `qb_floor50`.

Both evaluated on the same 8 real walk-forward folds (`MIN_TRAIN_SEASONS=9`, test seasons 2017–2024), both fully unconstrained (`monotone_constraints` all zero) — matching how `qb_model.json` itself is actually built, not the constrained variant that notebook's own diagnostic already rejected.

In [8]:
import xgboost as xgb
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 40
MAX_DEPTH_RANGE = (2, 8)  # widened -- no pre-assumed narrow band this time
MIN_TRAIN_SEASONS_AB = 9


def make_walk_forward_folds_4tuple(df, min_train_seasons=MIN_TRAIN_SEASONS_AB):
    # Same structure as 06a_model_qb.ipynb's fold generator (includes train_seasons,
    # needed by run_optuna_for_fold below) -- distinct from Section 3/6's 3-tuple
    # version, which RFECV's cv= parameter doesn't need.
    seasons_sorted = sorted(df["season"].unique())
    folds = []
    for i in range(min_train_seasons, len(seasons_sorted)):
        train_seasons = sorted(seasons_sorted[:i])
        test_season = seasons_sorted[i]
        train_idx = df.index[df["season"].isin(train_seasons)].to_numpy()
        test_idx = df.index[df["season"] == test_season].to_numpy()
        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append((train_idx, test_idx, test_season, train_seasons))
    return folds


class OptunaPruningCallback(xgb.callback.TrainingCallback):
    """Identical to 06a_model_qb.ipynb's version -- reads "validation_0", the
    XGBRegressor eval_set auto-name, not a custom "valid" key."""

    def __init__(self, trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log):
        score = evals_log["validation_0"]["mae"][-1]
        self.trial.report(score, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return False


def run_optuna_for_fold(df, train_idx, features, n_trials=N_TRIALS, seed=42):
    train_seasons_sorted = sorted(df.loc[train_idx, "season"].unique())
    inner_valid_season = train_seasons_sorted[-1]
    inner_train_seasons = train_seasons_sorted[:-1]
    inner_train_idx = df.index[df["season"].isin(inner_train_seasons) & df.index.isin(train_idx)]
    inner_valid_idx = df.index[(df["season"] == inner_valid_season) & df.index.isin(train_idx)]

    X_train = df.loc[inner_train_idx, features]
    y_train = df.loc[inner_train_idx, "vorp_next"]
    X_valid = df.loc[inner_valid_idx, features]
    y_valid = df.loc[inner_valid_idx, "vorp_next"]

    no_constraints = tuple(0 for _ in features)

    def objective(trial):
        params = {
            "objective": "reg:squarederror", "eval_metric": "mae",
            "max_depth": trial.suggest_int("max_depth", *MAX_DEPTH_RANGE),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "monotone_constraints": no_constraints, "seed": seed,
            "n_estimators": 500, "early_stopping_rounds": 20,
            "callbacks": [OptunaPruningCallback(trial)],
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
        trial.set_user_attr("best_iteration", model.best_iteration)
        return model.best_score

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study


def run_all_folds_ab(df, folds, features, label):
    fold_results, fold_best_params = [], []
    no_constraints = tuple(0 for _ in features)
    for tr, te, test_season, train_seasons in folds:
        study = run_optuna_for_fold(df, tr, features)
        best_params = dict(study.best_params)
        best_iteration = study.best_trial.user_attrs["best_iteration"]

        X_train_full = df.loc[tr, features]
        y_train_full = df.loc[tr, "vorp_next"]
        X_test = df.loc[te, features]
        final_model = xgb.XGBRegressor(
            objective="reg:squarederror", monotone_constraints=no_constraints, seed=42,
            n_estimators=max(best_iteration, 1), **best_params,
        )
        final_model.fit(X_train_full, y_train_full)
        preds = final_model.predict(X_test)
        actual = df.loc[te, "vorp_next"]

        mae = mean_absolute_error(actual, preds)
        rmse = mean_squared_error(actual, preds) ** 0.5
        rho = spearmanr(actual, preds)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan

        fold_results.append({"model": label, "test_season": test_season, "n_test": len(te), "mae": mae, "rmse": rmse, "spearman": rho})
        fold_best_params.append(best_params | {"n_estimators": best_iteration})

    fold_df = pd.DataFrame(fold_results)
    print(f"[{label}] Mean MAE: {fold_df['mae'].mean():.2f}, Mean RMSE: {fold_df['rmse'].mean():.2f}, Mean Spearman: {fold_df['spearman'].mean():.3f}")
    return fold_df, pd.DataFrame(fold_best_params)


folds_ab = make_walk_forward_folds_4tuple(qb)
print(f"{len(folds_ab)} walk-forward folds for the A/B comparison, test seasons {folds_ab[0][2]}-{folds_ab[-1][2]}")

MODEL_A_FEATURES = FEATURES_BASE
MODEL_B_FEATURES = FEATURES_BASE + ["td_rate_over_expected"]
print(f"\nModel A features ({len(MODEL_A_FEATURES)}): {MODEL_A_FEATURES}")
print(f"Model B features ({len(MODEL_B_FEATURES)}): {MODEL_B_FEATURES}")

fold_metrics_a, fold_params_a = run_all_folds_ab(qb, folds_ab, MODEL_A_FEATURES, label="Model A (shipped, 6 features)")
print(fold_metrics_a.to_string(index=False))

C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8 walk-forward folds for the A/B comparison, test seasons 2017-2024

Model A features (6): ['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'ol_pass_protection_proxy']
Model B features (7): ['scarcity_z', 'draft_pick_inverse', 'vorp_delta_yoy', 'passing_epa', 'age', 'ol_pass_protection_proxy', 'td_rate_over_expected']


[Model A (shipped, 6 features)] Mean MAE: 67.70, Mean RMSE: 88.02, Mean Spearman: 0.707
                        model  test_season  n_test       mae      rmse  spearman
Model A (shipped, 6 features)         2017      51 75.953759 97.022547  0.705882
Model A (shipped, 6 features)         2018      50 71.694487 99.889602  0.587673
Model A (shipped, 6 features)         2019      57 74.129699 89.847768  0.748935
Model A (shipped, 6 features)         2020      61 68.107018 85.551119  0.745267
Model A (shipped, 6 features)         2021      67 57.821716 78.595526  0.733931
Model A (shipped, 6 features)         2022      63 66.092642 87.391731  0.661850
Model A (shipped, 6 features)         2023      63 68.015463 86.522278  0.724807
Model A (shipped, 6 features)         2024      63 59.809192 79.359111  0.750786


In [9]:
fold_metrics_b, fold_params_b = run_all_folds_ab(qb_floor50, folds_ab, MODEL_B_FEATURES, label="Model B (+ td_rate_over_expected, floor=50)")
print(fold_metrics_b.to_string(index=False))

[Model B (+ td_rate_over_expected, floor=50)] Mean MAE: 67.88, Mean RMSE: 88.01, Mean Spearman: 0.704
                                      model  test_season  n_test       mae       rmse  spearman
Model B (+ td_rate_over_expected, floor=50)         2017      51 76.832028  96.612975  0.717647
Model B (+ td_rate_over_expected, floor=50)         2018      50 75.631458 103.059226  0.545655
Model B (+ td_rate_over_expected, floor=50)         2019      57 69.887259  86.277144  0.754517
Model B (+ td_rate_over_expected, floor=50)         2020      61 64.289804  83.572139  0.754839
Model B (+ td_rate_over_expected, floor=50)         2021      67 57.853902  81.153975  0.738155
Model B (+ td_rate_over_expected, floor=50)         2022      63 72.744170  94.339696  0.631632
Model B (+ td_rate_over_expected, floor=50)         2023      63 69.081702  83.835839  0.698021
Model B (+ td_rate_over_expected, floor=50)         2024      63 56.697198  75.246673  0.789223


In [10]:
print("Model A per-fold best hyperparameters (max_depth search range was 2-8):")
print(fold_params_a.drop(columns=["n_estimators"]).to_string(index=False))
print(f"\nModel A max_depth values chosen: {sorted(fold_params_a['max_depth'].tolist())}")
print(f"Any fold prefer depth > 2? {(fold_params_a['max_depth'] > 2).any()} "
      f"({(fold_params_a['max_depth'] > 2).sum()}/{len(fold_params_a)} folds)")

print("\nModel B per-fold best hyperparameters (max_depth search range was 2-8):")
print(fold_params_b.drop(columns=["n_estimators"]).to_string(index=False))
print(f"\nModel B max_depth values chosen: {sorted(fold_params_b['max_depth'].tolist())}")
print(f"Any fold prefer depth > 2? {(fold_params_b['max_depth'] > 2).any()} "
      f"({(fold_params_b['max_depth'] > 2).sum()}/{len(fold_params_b)} folds)")

comparison = pd.DataFrame([
    {"model": "Model A (shipped, 6 features)", "mae": fold_metrics_a["mae"].mean(),
     "spearman": fold_metrics_a["spearman"].mean(), "n_features": len(MODEL_A_FEATURES)},
    {"model": "Model B (+ td_rate_over_expected, floor=50)", "mae": fold_metrics_b["mae"].mean(),
     "spearman": fold_metrics_b["spearman"].mean(), "n_features": len(MODEL_B_FEATURES)},
])
print("\n" + comparison.round(3).to_string(index=False))

mae_win = fold_metrics_b["mae"].mean() < fold_metrics_a["mae"].mean()
spearman_win = fold_metrics_b["spearman"].mean() > fold_metrics_a["spearman"].mean()
if mae_win and spearman_win:
    verdict = "MODEL B WINS -- beats Model A on both MAE and Spearman simultaneously."
elif mae_win or spearman_win:
    verdict = "MIXED, NOT A CLEAR WIN -- Model B only beats Model A on one metric."
else:
    verdict = "MODEL A HOLDS -- Model B does not improve on either metric."
print(f"\nDecision rule verdict: {verdict}")

Model A per-fold best hyperparameters (max_depth search range was 2-8):
 max_depth  min_child_weight  reg_lambda  learning_rate  subsample
         3                 1    5.399484       0.077254   0.883229
         2                 3    0.282959       0.224738   0.727370
         2                 6    0.129194       0.224156   0.708329
         3                 5    3.721730       0.224915   0.750155
         2                 7    0.200096       0.271081   0.782888
         2                 8    5.741106       0.211085   0.665188
         7                 2    0.571611       0.217891   0.636554
         5                 7    1.227392       0.252914   0.623236

Model A max_depth values chosen: [2, 2, 2, 2, 3, 3, 5, 7]
Any fold prefer depth > 2? True (4/8 folds)

Model B per-fold best hyperparameters (max_depth search range was 2-8):
 max_depth  min_child_weight  reg_lambda  learning_rate  subsample
         4                 8    4.588018       0.223110   0.902719
         3     

## Section 7 verdict

**Does either model's folds now prefer something deeper than 2, with the search widened to 2–8?** Yes, decisively, for both:

- **Model A**: chosen `max_depth` per fold was `[2, 2, 2, 2, 3, 3, 5, 7]` — **4 of 8 folds** (half) preferred something deeper than 2, including two folds that went all the way to 5 and 7.
- **Model B**: chosen `max_depth` per fold was `[2, 2, 2, 3, 3, 3, 4, 4]` — **5 of 8 folds** preferred deeper than 2.

This directly confirms what the Section 7 intro flagged: RFECV's own feature-selection decisions in Sections 3 and 6 were made by a model artificially capped at `max_depth=3` (matching `05_feature_selection.ipynb`'s standard, confirmed explicit, not default). Roughly half of these real walk-forward folds, once actually given the room, wanted more depth than that. RFECV's selections aren't wrong because of this — but they were never given the chance to reflect what a deeper model would find useful.

**MAE / Spearman comparison, real held-out folds:**

| Model | MAE | Spearman | # features |
|---|---|---|---|
| Model A (shipped, 6 features) | 67.70 | 0.707 | 6 |
| Model B (+ `td_rate_over_expected`, floor=50) | 67.88 | 0.704 | 7 |

**Decision rule verdict: Model A holds.** Model B is *worse* on both metrics simultaneously — not mixed, not a rounding-error tie, worse on both. Despite RFECV picking `td_rate_over_expected` over `ol_pass_protection_proxy` in Section 6b, a real Optuna-tuned, properly-evaluated version of that swap does not produce a better model. **This resolves the open lead from Section 6 negatively**: `td_rate_over_expected` looked promising in RFECV's own selection process, but doesn't hold up once actually tuned and evaluated on genuine held-out folds. Worth naming plainly as the actual lesson here: RFECV's elimination order (built on a fixed, shallow, untuned estimator) is not the same question as "does this feature make a properly-tuned model better" — this experiment is a concrete, real example of those two things disagreeing, not just a hypothetical risk.

**One real, unplanned side-finding, separate from the Model A/B question**: Model A's own numbers here (67.70 MAE / 0.707 Spearman, `max_depth` searched 2–8) come out slightly *better* than the currently-shipped `qb_model.json`'s own official fold numbers from its last run (~68.13 MAE / 0.706 Spearman, `max_depth` searched only 2–4). That's not a claim this notebook is set up to validate rigorously — it's the same 6 features scored on the same folds, but not run under identical conditions elsewhere in this notebook (live-data pull timing differs, and this is the first time `qb_model.json`'s own feature set has ever been given a depth search wider than 2–4). It's a real, concrete signal that the shipped model's own `max_depth` search range — not just this experiment's new feature — may be worth revisiting deliberately, as its own separate, focused piece of work, not folded into this notebook's read-only scope.

**Stopping here.** Confirmed once more: `git status` shows no changes to `config/selected_features.yaml` or `data/models/qb_model.json` — this notebook remains fully read-only with respect to the shipped model.

## Section 8 — Winsorizing `vorp_delta_yoy`: does capping the extreme tail help?

Prompted directly by the two read-only diagnostics already run against the shipped `qb_model.json` (systematic bucket-MAE by `|vorp_delta_yoy|` across the whole training set, and SHAP on Maye/Stafford/Lawrence/Williams' 2026 inputs): both found real signal, not a mis-aimed suspicion. The bucket diagnostic showed MAE climbing from ~46 (small deltas) to ~67-71 (deltas beyond 100-200), while only ~5.7% of training rows ever see `|vorp_delta_yoy| > 200` — a genuinely thin, high-error region, not a big chunk of the data. SHAP confirmed `vorp_delta_yoy` is the single largest negative driver of all four flagged players' 2026 predictions (all four sit in that same thin tail), while Josh Allen — a normal delta — doesn't show the same effect.

This section builds a **winsorized** version of `vorp_delta_yoy` — the extreme tail clipped to a fixed bound, everything else left untouched — and runs the exact same real head-to-head as Section 7: same Optuna process (40 trials/fold, `max_depth` 2-8, TPE + median pruning), same 8 real walk-forward folds, same fully-unconstrained setup `qb_model.json` itself uses. **Model B only wins if it beats Model A on both MAE and Spearman simultaneously — the identical decision rule as every prior comparison in this notebook.**

**This remains fully read-only with respect to the shipped model, regardless of what this section finds.** Nothing here overwrites `data/models/qb_model.json` or `data/processed/fold_metrics_qb.csv` — a genuine win here is a flagged recommendation for a deliberate, separate follow-up, not something this notebook ships on its own.

In [11]:
# Decide the winsorization bound directly from this run's own distribution -- not a
# hardcoded number, so it moves with live data the same way everything else here does.
delta = qb["vorp_delta_yoy"].dropna()
print(f"vorp_delta_yoy -- {len(delta)} rows with a usable value (of {len(qb)} total)")
print("\nSigned percentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p:>2}: {delta.quantile(p / 100):8.2f}")

q3_q4_boundary = delta.abs().quantile(0.75)  # the boundary the qb_model.json bucket-MAE diagnostic used
cap_lower = delta.quantile(0.05)
cap_upper = delta.quantile(0.95)
print(f"\nQ3/Q4 boundary on |delta| (from the prior bucket-MAE diagnostic): +/-{q3_q4_boundary:.1f}")
print(f"  -> would clip {(delta.abs() > q3_q4_boundary).sum()} / {len(delta)} rows ({(delta.abs() > q3_q4_boundary).mean():.1%})")
print(f"5th/95th percentile bound (chosen below): [{cap_lower:.1f}, {cap_upper:.1f}]")
print(f"  -> would clip {((delta < cap_lower) | (delta > cap_upper)).sum()} / {len(delta)} rows "
      f"({((delta < cap_lower) | (delta > cap_upper)).mean():.1%})")

vorp_delta_yoy -- 732 rows with a usable value (of 980 total)

Signed percentiles:
  p 1:  -240.73
  p 5:  -151.98
  p10:  -107.57
  p25:   -58.70
  p50:    -7.65
  p75:    42.32
  p90:   102.99
  p95:   154.06
  p99:   259.94

Q3/Q4 boundary on |delta| (from the prior bucket-MAE diagnostic): +/-96.3
  -> would clip 183 / 732 rows (25.0%)
5th/95th percentile bound (chosen below): [-152.0, 154.1]
  -> would clip 74 / 732 rows (10.1%)


**Bound chosen: the 5th/95th percentile of `vorp_delta_yoy`'s own real distribution, not the Q3/Q4 boundary.** The Q3/Q4 boundary (~±96, the same one the prior bucket-MAE diagnostic used to define its top quartile) would clip roughly a quarter of all rows with a usable delta — that's not targeting an extreme tail, that's reshaping the top quartile of what's otherwise a real, mostly well-behaved signal. The 5th/95th percentile bound clips only the rarest ~10% of rows (5% each side), which lines up much better with where the bucket-MAE diagnostic actually found the problem: MAE was still unremarkable through Q3 (deltas up to ~96, MAE 62.19) and only got clearly worse **beyond ~150-200** (MAE 66.60 at >150, 71.15 at >200) — exactly the region a 5th/95th cap targets, not the broader top quartile. Winsorizing at Q3/Q4 risks throwing away real signal from a lot of ordinary players to fix a problem that's actually concentrated much further out in the tail.

In [12]:
qb = qb.assign(vorp_delta_yoy_capped=qb["vorp_delta_yoy"].clip(lower=cap_lower, upper=cap_upper))
# NaN != NaN evaluates True (IEEE-754), so a naive != comparison would count every row with a
# missing delta as "clipped" even though both sides are still NaN -- guard with notna() first.
clipped_mask = qb["vorp_delta_yoy"].notna() & (qb["vorp_delta_yoy"] != qb["vorp_delta_yoy_capped"])
n_clipped = clipped_mask.sum()
print(f"Rows actually clipped: {n_clipped} / {len(qb)} ({n_clipped / len(qb):.1%})")
print(qb.loc[clipped_mask, ["player_display_name", "season", "vorp_delta_yoy", "vorp_delta_yoy_capped"]]
      .sort_values("vorp_delta_yoy").to_string(index=False))

# Same 6 features, same column name -- just swapping in the capped values for the head-to-head below.
qb_capped_df = qb.assign(vorp_delta_yoy=qb["vorp_delta_yoy_capped"])

Rows actually clipped: 74 / 980 (7.6%)
player_display_name  season  vorp_delta_yoy  vorp_delta_yoy_capped
     Jameis Winston    2020         -356.49               -151.979
 Ben Roethlisberger    2019         -319.35               -151.979
      Aaron Rodgers    2023         -259.90               -151.979
         Cam Newton    2019         -253.65               -151.979
    Jacoby Brissett    2018         -246.92               -151.979
      Tyler Thigpen    2009         -246.49               -151.979
          Tony Romo    2015         -243.91               -151.979
        Andrew Luck    2015         -241.17               -151.979
       Daniel Jones    2023         -239.74               -151.979
      Aaron Rodgers    2017         -237.91               -151.979
       Dak Prescott    2024         -237.77               -151.979
       Dak Prescott    2020         -233.17               -151.979
    Chad Pennington    2009         -232.69               -151.979
    Jacoby Brissett    

### Head-to-head: Model A (shipped, uncapped) vs. Model B (winsorized `vorp_delta_yoy`)

**Model A** reuses Section 7's own run above unchanged — same 6 features, same folds, same fully-unconstrained Optuna search, nothing about it differs in this section. **Model B** is the identical 6-feature spec, identical process, with only `vorp_delta_yoy` replaced by its winsorized version.

In [13]:
print("Model A (shipped spec, uncapped vorp_delta_yoy) -- reusing Section 7's run, unchanged:")
print(fold_metrics_a[["test_season", "mae", "spearman"]].to_string(index=False))
print(f"[Model A] Mean MAE: {fold_metrics_a['mae'].mean():.2f}, Mean Spearman: {fold_metrics_a['spearman'].mean():.3f}")

fold_metrics_delta_capped, fold_params_delta_capped = run_all_folds_ab(
    qb_capped_df, folds_ab, MODEL_A_FEATURES,
    label=f"Model B (winsorized vorp_delta_yoy, [{cap_lower:.0f}, {cap_upper:.0f}])",
)
print(fold_metrics_delta_capped.to_string(index=False))

Model A (shipped spec, uncapped vorp_delta_yoy) -- reusing Section 7's run, unchanged:
 test_season       mae  spearman
        2017 75.953759  0.705882
        2018 71.694487  0.587673
        2019 74.129699  0.748935
        2020 68.107018  0.745267
        2021 57.821716  0.733931
        2022 66.092642  0.661850
        2023 68.015463  0.724807
        2024 59.809192  0.750786
[Model A] Mean MAE: 67.70, Mean Spearman: 0.707


[Model B (winsorized vorp_delta_yoy, [-152, 154])] Mean MAE: 68.78, Mean RMSE: 89.56, Mean Spearman: 0.699
                                           model  test_season  n_test       mae       rmse  spearman
Model B (winsorized vorp_delta_yoy, [-152, 154])         2017      51 71.485101  92.267406  0.750317
Model B (winsorized vorp_delta_yoy, [-152, 154])         2018      50 73.548334 100.137141  0.581046
Model B (winsorized vorp_delta_yoy, [-152, 154])         2019      57 71.569663  92.070398  0.747926
Model B (winsorized vorp_delta_yoy, [-152, 154])         2020      61 70.694535  88.785730  0.737229
Model B (winsorized vorp_delta_yoy, [-152, 154])         2021      67 61.233784  84.302879  0.679583
Model B (winsorized vorp_delta_yoy, [-152, 154])         2022      63 71.813586  92.766395  0.637040
Model B (winsorized vorp_delta_yoy, [-152, 154])         2023      63 65.668727  82.540820  0.766186
Model B (winsorized vorp_delta_yoy, [-152, 154])         2024      63 64.200821  83.6

In [14]:
comparison_8 = pd.DataFrame([
    {"model": "Model A (shipped, uncapped vorp_delta_yoy)",
     "mae": fold_metrics_a["mae"].mean(), "spearman": fold_metrics_a["spearman"].mean()},
    {"model": "Model B (winsorized vorp_delta_yoy)",
     "mae": fold_metrics_delta_capped["mae"].mean(), "spearman": fold_metrics_delta_capped["spearman"].mean()},
])
print(comparison_8.round(3).to_string(index=False))

mae_win = fold_metrics_delta_capped["mae"].mean() < fold_metrics_a["mae"].mean()
spearman_win = fold_metrics_delta_capped["spearman"].mean() > fold_metrics_a["spearman"].mean()
if mae_win and spearman_win:
    verdict_8 = "MODEL B WINS -- beats Model A on both MAE and Spearman simultaneously."
elif mae_win or spearman_win:
    verdict_8 = "MIXED, NOT A CLEAR WIN -- Model B only beats Model A on one metric."
else:
    verdict_8 = "MODEL A HOLDS -- Model B does not improve on either metric."
print(f"\nDecision rule verdict: {verdict_8}")

                                     model    mae  spearman
Model A (shipped, uncapped vorp_delta_yoy) 67.703     0.707
       Model B (winsorized vorp_delta_yoy) 68.777     0.699

Decision rule verdict: MODEL A HOLDS -- Model B does not improve on either metric.


### Does capping actually pull Maye/Stafford/Lawrence/Williams back toward reasonable values?

Builds a Model A final model (median hyperparameters, uncapped delta) and a Model B final model (median hyperparameters, winsorized delta) using the exact same recipe `06a_model_qb.ipynb` uses for `qb_model.json` itself — early-stopped against 2024, then refit on all labeled data. **Neither is saved anywhere; this is purely for the side-by-side comparison below.** Both are then fed the four flagged players' real 2025 feature rows (the same inputs behind the live 2026 predictions), plus Josh Allen as the well-behaved baseline.

In [15]:
# 06a_model_qb.ipynb's own pipeline drops the 2025 rows before this point (no vorp_next yet) --
# rebuild fresh here, read-only, purely to get their real 2025 feature rows for this comparison.
qb_all_with_2025 = build_qb_features_no_dropna(td_rate_attempts_floor=100)
target_players = ["Drake Maye", "Matthew Stafford", "Trevor Lawrence", "Caleb Williams", "Josh Allen"]
qb_2025_rows = qb_all_with_2025[qb_all_with_2025["player_display_name"].isin(target_players)].copy()
qb_2025_rows = qb_2025_rows[qb_2025_rows["season"] == 2025].copy()
qb_2025_rows["vorp_delta_yoy_capped"] = qb_2025_rows["vorp_delta_yoy"].clip(lower=cap_lower, upper=cap_upper)
print(qb_2025_rows[["player_display_name", "vorp_delta_yoy", "vorp_delta_yoy_capped"]].to_string(index=False))


def build_final_model_readonly(train_df, features, fold_params_df, monotone_constraints):
    """Same recipe as 06a_model_qb.ipynb's final-model cell -- median hyperparameters across
    folds, early-stopped against the last labeled season, refit on all labeled data. Returns
    the model object only; never calls save_model()."""
    stable_params = {
        "max_depth": int(round(fold_params_df["max_depth"].median())),
        "min_child_weight": int(round(fold_params_df["min_child_weight"].median())),
        "reg_lambda": float(fold_params_df["reg_lambda"].median()),
        "learning_rate": float(fold_params_df["learning_rate"].median()),
        "subsample": float(fold_params_df["subsample"].median()),
    }
    all_seasons_sorted = sorted(train_df["season"].unique())
    final_valid_season = all_seasons_sorted[-1]
    final_train_seasons = all_seasons_sorted[:-1]
    final_train_idx = train_df.index[train_df["season"].isin(final_train_seasons)]
    final_valid_idx = train_df.index[train_df["season"] == final_valid_season]

    probe_model = xgb.XGBRegressor(
        objective="reg:squarederror", eval_metric="mae", monotone_constraints=monotone_constraints, seed=42,
        n_estimators=500, early_stopping_rounds=20, **stable_params,
    )
    probe_model.fit(
        train_df.loc[final_train_idx, features], train_df.loc[final_train_idx, "vorp_next"],
        eval_set=[(train_df.loc[final_valid_idx, features], train_df.loc[final_valid_idx, "vorp_next"])],
        verbose=False,
    )
    final_n_estimators = max(probe_model.best_iteration, 1)

    final_model = xgb.XGBRegressor(
        objective="reg:squarederror", monotone_constraints=monotone_constraints, seed=42,
        n_estimators=final_n_estimators, **stable_params,
    )
    final_model.fit(train_df[features], train_df["vorp_next"])
    return final_model, stable_params, final_n_estimators


no_constraints_6 = tuple(0 for _ in MODEL_A_FEATURES)
model_a_final, params_a_final, n_est_a_final = build_final_model_readonly(qb, MODEL_A_FEATURES, fold_params_a, no_constraints_6)
model_b_final, params_b_final, n_est_b_final = build_final_model_readonly(qb_capped_df, MODEL_A_FEATURES, fold_params_delta_capped, no_constraints_6)
print(f"\nModel A final hyperparameters: {params_a_final}, n_estimators={n_est_a_final}")
print(f"Model B final hyperparameters: {params_b_final}, n_estimators={n_est_b_final}")

pred_a = model_a_final.predict(qb_2025_rows[MODEL_A_FEATURES])
qb_2025_rows_b_input = qb_2025_rows.copy()
qb_2025_rows_b_input["vorp_delta_yoy"] = qb_2025_rows_b_input["vorp_delta_yoy_capped"]
pred_b = model_b_final.predict(qb_2025_rows_b_input[MODEL_A_FEATURES])

result_8 = pd.DataFrame({
    "player": qb_2025_rows["player_display_name"].to_numpy(),
    "actual_2025_vorp": qb_2025_rows["vorp"].to_numpy(),
    "raw_vorp_delta_yoy": qb_2025_rows["vorp_delta_yoy"].to_numpy(),
    "capped_vorp_delta_yoy": qb_2025_rows["vorp_delta_yoy_capped"].to_numpy(),
    "predicted_2026_model_A_uncapped": pred_a,
    "predicted_2026_model_B_capped": pred_b,
})
result_8["shift_from_capping"] = result_8["predicted_2026_model_B_capped"] - result_8["predicted_2026_model_A_uncapped"]
result_8 = result_8.set_index("player").loc[target_players].reset_index()
print("\n" + result_8.round(1).to_string(index=False))

player_display_name  vorp_delta_yoy  vorp_delta_yoy_capped
   Matthew Stafford          152.48                152.480
         Josh Allen            6.26                  6.260
    Trevor Lawrence          214.66                154.062
         Drake Maye          189.00                154.062
     Caleb Williams           80.82                 80.820

Model A final hyperparameters: {'max_depth': 2, 'min_child_weight': 6, 'reg_lambda': 0.8995011828218823, 'learning_rate': 0.22444699748448305, 'subsample': 0.7178493269772277}, n_estimators=31
Model B final hyperparameters: {'max_depth': 2, 'min_child_weight': 4, 'reg_lambda': 1.4646318876824407, 'learning_rate': 0.23525177928876753, 'subsample': 0.6893939543268299}, n_estimators=42

          player  actual_2025_vorp  raw_vorp_delta_yoy  capped_vorp_delta_yoy  predicted_2026_model_A_uncapped  predicted_2026_model_B_capped  shift_from_capping
      Drake Maye              90.7               189.0                  154.1                   

## Section 8 verdict

**Model A holds -- winsorizing `vorp_delta_yoy` at the 5th/95th percentile makes the model worse, not better, on both metrics.**

| Model | MAE | Spearman |
|---|---|---|
| Model A (shipped, uncapped `vorp_delta_yoy`) | 67.70 | 0.707 |
| Model B (winsorized `vorp_delta_yoy`, [-152, 154]) | 68.78 | 0.699 |

Model B loses on both MAE (+1.6%) and Spearman (-0.008) simultaneously -- not a rounding-error tie, not mixed. Per the standing decision rule, **Model A holds**; this experiment does not recommend capping `vorp_delta_yoy` for the shipped model.

**Does capping pull the four flagged players back toward more reasonable predictions? No -- for two of them it makes things clearly worse.**

| Player | actual 2025 VORP | raw delta | capped delta | pred 2026 (A, uncapped) | pred 2026 (B, capped) | shift |
|---|---|---|---|---|---|---|
| Drake Maye | 90.7 | 189.0 | 154.1 (clipped) | +11.1 | **-17.9** | -29.0 |
| Matthew Stafford | 89.6 | 152.5 | 152.5 (not clipped) | -7.0 | **-39.5** | -32.5 |
| Trevor Lawrence | 81.4 | 214.7 | 154.1 (clipped) | -50.0 | -48.7 | +1.3 |
| Caleb Williams | 55.9 | 80.8 | 80.8 (not clipped) | -10.6 | -14.7 | -4.1 |
| Josh Allen (baseline) | 105.9 | 6.3 | 6.3 (not clipped) | +67.8 | +66.6 | -1.1 |

Capping did **not** rescue the two players it actually touched. Maye's prediction, already a modest positive number under the uncapped model (+11.1), turns *more negative* under the capped model (-17.9) -- the opposite of what the extrapolation hypothesis predicted. Stafford's prediction, already negative, gets substantially *more* negative (-7.0 to -39.5). Only Trevor Lawrence -- whose delta was clipped by the largest amount (214.7 to 154.1) -- barely moves (+1.3), and it's still deeply negative either way. Caleb Williams' delta (80.8) falls *below* the 95th-percentile cap and isn't clipped at all, yet his prediction still shifts (-4.1) -- a reminder that capping one feature changes the whole Optuna-refit tree structure, not just the clipped rows' own predictions. Josh Allen, the well-behaved baseline whose delta was never near the cap, moves by only -1.1 -- noise from refitting with different median hyperparameters, not a real capping effect.

**Bottom line**: the two prior diagnostics were right about *where* the problem lives (large `vorp_delta_yoy`, thin data, real elevated error) but this experiment shows winsorizing doesn't fix *what's wrong*. It doesn't help held-out accuracy, and it doesn't move the specific mispredictions in the direction the extrapolation hypothesis predicted -- Maye and Stafford get *worse*, not better. Whatever is driving those two specific misses, it isn't simply "the raw delta value is too extreme for the tree to fit" -- clipping the value away doesn't fix it and measurably hurts the model elsewhere. Reported here as a negative result; no follow-up variation (different cap bound, log-transform, etc.) is chased without a new decision point, per this notebook's standing discipline.

**Confirmed per the read-only requirement**: this section made zero writes to `config/selected_features.yaml` and zero writes to `data/models/qb_model.json` / `data/processed/fold_metrics_qb.csv` -- `git status` after running shows no changes to either path. Both final models built above (`model_a_final`, `model_b_final`) exist only in this notebook's memory; neither was saved anywhere. QB's shipped model is unaffected by this experiment.

## Section 9 -- Historical validation: does the shipped model's "big delta -> predict decline" pattern actually hold up on real, already-known outcomes?

Section 8 showed winsorizing `vorp_delta_yoy` doesn't help, and specifically doesn't rescue Maye's or Stafford's 2026 predictions -- if anything it makes them worse. That leaves the real, un-resolved question: is the model's tendency to predict a decline for large-delta players a *genuine, generalizable* pattern (the same kind of TD-rate mean-reversion story documented for Stafford-like cases in the public literature), or is it overfit noise that happens to fire on Maye/Stafford/Lawrence without being a real signal?

This is directly testable, because every prior season already happened: **every historical QB row with `vorp_delta_yoy > 100`** (matching the range Maye 189.0 / Stafford 152.5 / Lawrence 214.7 sit in) already has a real, known `vorp_next` -- there's nothing to wait for. For each one, this section shows what `qb_model.json` (the actual shipped artifact, unmodified) predicts, and what actually happened.

**Read this with one honest caveat up front**: `qb_model.json` was fit on these same 980 rows, so this is an in-sample check, not a held-out one -- the model has literally seen every one of these outcomes during training. That means a *good* result here is weaker evidence than it looks (the model could just be memorizing), but a *bad* result is strong evidence regardless -- if the model can't even fit its own training data well in this region, that's a real structural problem, not something a favorable framing can explain away. This section stays fully read-only; nothing here touches `qb_model.json` or `config/selected_features.yaml`.

In [16]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

shipped_model = xgb.XGBRegressor()
shipped_model.load_model(str(REPO_ROOT / "data" / "models" / "qb_model.json"))

overall_pred = shipped_model.predict(qb[FEATURES_BASE])
overall_mae_in_sample = mean_absolute_error(qb["vorp_next"], overall_pred)
print(f"qb_model.json in-sample MAE across all {len(qb)} training rows: {overall_mae_in_sample:.2f}")
print("(for reference: qb_model.json's honest, held-out walk-forward MAE from fold_metrics_qb.csv is 67.72 --")
print(" in-sample error is normally *lower* than held-out error, since the model was fit on these exact rows)")

extreme_hist = qb[qb["vorp_delta_yoy"] > 100].copy()
print()
print(f"Historical rows with vorp_delta_yoy > 100: {len(extreme_hist)} / {len(qb)} ({len(extreme_hist) / len(qb):.1%})")

extreme_hist["predicted_vorp_next"] = shipped_model.predict(extreme_hist[FEATURES_BASE])
extreme_hist["error"] = extreme_hist["predicted_vorp_next"] - extreme_hist["vorp_next"]
extreme_hist["abs_error"] = extreme_hist["error"].abs()
extreme_hist = extreme_hist.sort_values("abs_error").reset_index(drop=True)

print(f"Extreme-delta bucket in-sample MAE: {extreme_hist['abs_error'].mean():.2f} "
      f"(vs. {overall_mae_in_sample:.2f} overall in-sample, vs. 67.72 overall held-out)")

pd.set_option("display.max_rows", None)
print()
print(f"All {len(extreme_hist)} rows, sorted by absolute error (best-fit analogs first, worst last):")
print(extreme_hist[["player_display_name", "season", "vorp_delta_yoy", "predicted_vorp_next", "vorp_next", "error", "abs_error"]]
      .round(1).to_string(index=False))

qb_model.json in-sample MAE across all 980 training rows: 55.61
(for reference: qb_model.json's honest, held-out walk-forward MAE from fold_metrics_qb.csv is 67.72 --
 in-sample error is normally *lower* than held-out error, since the model was fit on these exact rows)

Historical rows with vorp_delta_yoy > 100: 81 / 980 (8.3%)
Extreme-delta bucket in-sample MAE: 70.09 (vs. 55.61 overall in-sample, vs. 67.72 overall held-out)

All 81 rows, sorted by absolute error (best-fit analogs first, worst last):
player_display_name  season  vorp_delta_yoy  predicted_vorp_next  vorp_next  error  abs_error
    Gardner Minshew    2023           156.7          -178.100006     -179.1    1.1        1.1
      Aaron Rodgers    2014           172.1            19.100000       16.3    2.8        2.8
          Matt Ryan    2016           123.9           -12.400000      -20.0    7.6        7.6
         Joe Flacco    2016           102.9           -69.800003      -62.1   -7.7        7.7
    Patrick Mahomes    

In [17]:
from scipy.stats import spearmanr

n = len(extreme_hist)
pred_neg = extreme_hist["predicted_vorp_next"] < 0
actual_neg = extreme_hist["vorp_next"] < 0

print(f"Of {n} historical rows with vorp_delta_yoy > 100:")
print(f"  Model predicted a decline (predicted_vorp_next < 0): {pred_neg.sum()} / {n} ({pred_neg.mean():.1%})")
print(f"  -> of those {pred_neg.sum()} decline calls: {(pred_neg & actual_neg).sum()} were actually right "
      f"(player really did decline), {(pred_neg & ~actual_neg).sum()} were wrong (player actually improved)")
print(f"  Model predicted improvement (predicted_vorp_next >= 0): {(~pred_neg).sum()} / {n}")
print(f"  -> of those: {(~pred_neg & ~actual_neg).sum()} were right, {(~pred_neg & actual_neg).sum()} were wrong")

rho, _ = spearmanr(extreme_hist["predicted_vorp_next"], extreme_hist["vorp_next"])
print()
print(f"Spearman correlation within this bucket alone (predicted vs. actual): {rho:.3f}")
print(f"Median absolute error within this bucket: {extreme_hist['abs_error'].median():.2f}")
print(f"Rows with abs_error > 100 (a genuinely large miss): {(extreme_hist['abs_error'] > 100).sum()} / {n} "
      f"({(extreme_hist['abs_error'] > 100).mean():.1%})")

Of 81 historical rows with vorp_delta_yoy > 100:
  Model predicted a decline (predicted_vorp_next < 0): 74 / 81 (91.4%)
  -> of those 74 decline calls: 55 were actually right (player really did decline), 19 were wrong (player actually improved)
  Model predicted improvement (predicted_vorp_next >= 0): 7 / 81
  -> of those: 5 were right, 2 were wrong

Spearman correlation within this bucket alone (predicted vs. actual): 0.696
Median absolute error within this bucket: 64.47
Rows with abs_error > 100 (a genuinely large miss): 19 / 81 (23.5%)


## Section 9 verdict

**Neither a clean confirmation nor a clean refutation -- a real, moderate, majority-right pattern with unreliable calibration, and one genuinely damning structural signal.**

**The damning signal first**: this bucket's in-sample MAE (70.09) is *worse* than `qb_model.json`'s own honest, held-out walk-forward MAE (67.72) -- despite being scored on rows the model was directly fit on. In-sample error is supposed to be the *easy* number (the whole-dataset in-sample MAE is 55.61, a full 12 points better than the held-out estimate, exactly as expected). For the extreme-delta rows specifically, that relationship inverts: the model can't even fit its own training examples in this region as well as it generalizes everywhere else. That's a real structural weak spot, not just ordinary extrapolation risk on unseen data.

**But the directional pattern isn't pure noise, either.** The model predicts a decline (`predicted_vorp_next < 0`) for 74 of these 81 rows (91.4%) -- it essentially always calls for reversion once delta is large, rather than differentiating case by case. Of those 74 decline calls, 55 (74.3%) were actually right; 19 (25.7%) were wrong -- the player genuinely improved instead. Across the whole bucket, the model's directional call was right 60/81 times (74.1%) and wrong 21/81 times (25.9%). That's clearly better than a coin flip, and the within-bucket Spearman correlation (0.696) is close to the model's own overall official Spearman (0.700) -- rank-ordering ability is essentially intact here, not destroyed.

**What's unreliable is magnitude, not direction.** Median absolute error in this bucket is 64.47, and 19/81 rows (23.5%) miss by more than 100 points -- including some spectacular misses in both directions: Jameis Winston's 2019 season (predicted -57.7, actual -290.3, off by 232.6 -- the single worst case) shows the model *underestimating* how bad a decline could get, while Jalen Hurts 2021 (predicted -55.4, actual +117.7, off by 173.2) and Josh Allen 2019 (predicted -54.3, actual +112.8, off by 167.2) show it confidently calling for a decline that never happened. Against that same backdrop, some analogs land almost exactly (Gardner Minshew 2023: off by 1.1; Aaron Rodgers 2014: off by 2.8) -- there's no reliable way to tell which kind of case a new one will be from the feature values alone.

**What this means for Maye (+11.1), Stafford (-7.0), and Lawrence (-50.0)**: none of these three predictions should be read as confident, well-calibrated numbers. Lawrence's -50.0 sits squarely in the historically-typical "predicted decline" range, and history says that call is right about 3 times in 4 -- better than nothing, but a real 1-in-4 chance it's simply wrong, plus historically-typical errors of 50-150+ points even when the direction is right. Maye's mild +11.1 and Stafford's mild -7.0 are actually less extreme than most of this bucket's predictions (which cluster heavily negative), for reasons the model's internals don't make legible from here. **Bottom line**: the "big delta predicts decline" behavior is a real, majority-right tendency baked into the model from genuine historical patterns -- it is not simply overfit noise with zero signal -- but it is nowhere near reliable enough to trust as a precise point estimate for any individual player, and this specific region is demonstrably where the model fits worst, even against its own training data.

**Confirmed per the read-only requirement**: this section made zero writes to `config/selected_features.yaml` and zero writes to `data/models/qb_model.json` / `data/processed/fold_metrics_qb.csv` -- `git status` after running shows no changes to either path. `qb_model.json` was only loaded and used for prediction, never refit or saved.